# Module 1 — Language Detection

TF-IDF (character n-grams) + Logistic Regression over `papluca/language-identification` (90k samples, 20 languages, pre-split).

**Why char n-grams, not word n-grams**: language ID is about *how words are spelled*, not vocabulary — short character sequences separate languages cleanly even on short, informal text like a customer message.

In [ ]:
import sys
sys.path.append('..')
from datasets import load_dataset
import pandas as pd
from src.language_detection.train import build_pipeline
from src import config

## Load & inspect the data

In [ ]:
ds = load_dataset(config.LANG_DATASET)
train, val, test = ds['train'], ds['validation'], ds['test']
print(train)
pd.Series(train['labels']).value_counts()

## Train

In [ ]:
pipe = build_pipeline()
pipe.fit(train['text'], train['labels'])

## Evaluate

In [ ]:
from sklearn.metrics import classification_report, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

preds = pipe.predict(test['text'])
print(classification_report(test['labels'], preds))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))
ConfusionMatrixDisplay.from_predictions(test['labels'], preds, ax=ax, xticks_rotation=90)
plt.title('Language ID — confusion matrix')
plt.tight_layout()
plt.show()

## Save the model

Run the equivalent standalone script instead if you just want the artifact:
```bash
python -m src.language_detection.train
```

In [ ]:
import joblib
joblib.dump(pipe, config.LANG_MODEL_PATH)
print('Saved to', config.LANG_MODEL_PATH)

## Spot-check

In [ ]:
from src.language_detection.predict import detect_language
for s in ['Where is my order?', '¿Dónde está mi pedido?', 'Où est ma commande?']:
    print(s, '->', detect_language(s))